In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, make_scorer

# =================================================================================
# ORIGINAL CUSTOM SCORER (AUS DEM HAUPTBERICHT)
# =================================================================================
def bike_custom_score(y_true, y_pred):
    """
    Asymmetric Weighted Business Score.
    Das ist die "strenge" Version aus dem Hauptprojekt.
    """
    y_true = np.array(y_true).astype(float)
    y_pred = np.array(y_pred).astype(float)
    
    # Business-Grenzen
    lower = y_true * 0.95
    upper = y_true * 1.20
    
    # Basis: Alles ist erstmal 1.0 (Perfekt)
    score = np.ones_like(y_true, dtype=float)
    
    # Masken
    in_band = (y_pred >= lower) & (y_pred <= upper)
    outside = ~in_band
    under = y_pred < lower
    over = y_pred > upper
    
    # Relativer Fehler
    err_ratio = np.abs(y_pred - y_true) / (y_true + 1e-9)
    
    # Gewichtung (Wurzel der Nachfrage) - Wichtig für Rush Hour!
    weight = np.sqrt(y_true)
    weight = weight / (np.mean(weight) + 1e-9) # Normalisieren
    
    # Strenge Penalties
    penalty_under = 2.5 * err_ratio * weight  # Faktor 2.5 (teuer)
    penalty_over = 0.8 * err_ratio * weight   # Faktor 0.8 (billig)
    
    # Abzug
    score[outside & under] -= penalty_under[outside & under]
    score[outside & over] -= penalty_over[outside & over]
    
    # Clip auf 0 (keine negativen Scores)
    score = np.clip(score, 0.0, 1.0)
    
    return float(np.mean(score))

# Scorer-Objekt für GridSearchCV
from sklearn.metrics import make_scorer
bike_scorer = make_scorer(bike_custom_score, greater_is_better=True)



def bike_score_simple(y_true, y_pred):
    """Simple Binary Score (für Vergleich)"""
    y_true = np.array(y_true).astype(float)
    y_pred = np.array(y_pred).astype(float)
    lower = y_true * 0.95
    upper = y_true * 1.20
    ok = (y_pred >= lower) & (y_pred <= upper)
    return np.mean(ok) * 100

bike_scorer = make_scorer(bike_custom_score, greater_is_better=True)

In [2]:
# =================================================================================
# DATEN LADEN (ANPASSEN AN DEINEN PFAD)
# =================================================================================
df = pd.read_csv("london_merged.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Streiktage entfernen
strike_dates = ['2015-07-09', '2015-08-06']
df = df[~df['timestamp'].dt.strftime('%Y-%m-%d').isin(strike_dates)]

# Feature Engineering (gleich wie im Hauptcode)
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

# Split
df = df.sort_values('timestamp').reset_index(drop=True)
target = 'cnt'
features = [c for c in df.columns if c not in ['cnt', 'timestamp']]

split_index = int(len(df) * 0.8)
X_train = df.iloc[:split_index][features]
y_train = df.iloc[:split_index][target]
X_test = df.iloc[split_index:][features]
y_test = df.iloc[split_index:][target]

print(f"Training: {len(X_train)} | Test: {len(X_test)}")

Training: 13892 | Test: 3474


In [3]:
# =================================================================================
# PARAMETER GRID (ERWEITERT)
# =================================================================================
param_configs = [
    # Baseline (schnell)
    {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'min_samples_leaf': 1, 'subsample': 1.0},
    
    # Deine bisherige beste Konfiguration
    {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 500, 'min_samples_leaf': 4, 'subsample': 0.8},
    
    # Konservativer (verhindert Underestimation)
    {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 500, 'min_samples_leaf': 2, 'subsample': 0.8},
    
    # Tiefer (komplexere Muster)
    {'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 1000, 'min_samples_leaf': 4, 'subsample': 0.8},
    
    # Aggressiv (schneller lernen)
    {'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 200, 'min_samples_leaf': 2, 'subsample': 1.0},
]


In [4]:
# =================================================================================
# MANUELLE GRID-SEARCH MIT CUSTOM SCORER
# =================================================================================
print("\n--- Starte Hyperparameter-Optimierung für GradientBoostingRegressor ---")
print("OPTIMIERT AUF: Asymmetric Weighted Score (nicht R²)\n")

tscv = TimeSeriesSplit(n_splits=5)
best_score = -np.inf
best_params = None
results = []

for config in param_configs:
    print(f"Teste Konfiguration: {config} ...")
    
    # Modell erstellen
    gb = GradientBoostingRegressor(
        random_state=42,
        **config
    )
    
    # Cross-Validation mit CUSTOM SCORER
    cv_scores = []
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        gb.fit(X_tr, y_tr)
        y_pred_val = gb.predict(X_val)
        score = bike_custom_score(y_val, y_pred_val)
        cv_scores.append(score)
    
    avg_score = np.mean(cv_scores)
    print(f" -> Avg CV Weighted Score: {avg_score:.4f}")
    
    results.append({
        'config': config,
        'cv_weighted_score': avg_score
    })
    
    if avg_score > best_score:
        best_score = avg_score
        best_params = config

print(f"\n🏆 GEWINNER KONFIGURATION: {best_params}")
print(f"Bester CV Weighted Score: {best_score:.4f}")


--- Starte Hyperparameter-Optimierung für GradientBoostingRegressor ---
OPTIMIERT AUF: Asymmetric Weighted Score (nicht R²)

Teste Konfiguration: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'min_samples_leaf': 1, 'subsample': 1.0} ...
 -> Avg CV Weighted Score: 0.7043
Teste Konfiguration: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 500, 'min_samples_leaf': 4, 'subsample': 0.8} ...
 -> Avg CV Weighted Score: 0.7383
Teste Konfiguration: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 500, 'min_samples_leaf': 2, 'subsample': 0.8} ...
 -> Avg CV Weighted Score: 0.7449
Teste Konfiguration: {'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 1000, 'min_samples_leaf': 4, 'subsample': 0.8} ...
 -> Avg CV Weighted Score: 0.7524
Teste Konfiguration: {'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 200, 'min_samples_leaf': 2, 'subsample': 1.0} ...
 -> Avg CV Weighted Score: 0.7049

🏆 GEWINNER KONFIGURATION: {'learning_rate': 0.01, 'max_depth': 6, 'n_

In [5]:
# =================================================================================
# FINALES TRAINING UND EVALUATION
# =================================================================================
print("\n--- Finales Training auf gesamten Trainingsdaten ---")
final_gb = GradientBoostingRegressor(random_state=42, **best_params)
final_gb.fit(X_train, y_train)

y_pred_test = final_gb.predict(X_test)

# Alle Metriken berechnen
r2 = r2_score(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
weighted_score = bike_custom_score(y_test, y_pred_test)
simple_score = bike_score_simple(y_test, y_pred_test)

print("\n=== FINALER TEST REPORT ===")
print(f"R² Score:                {r2:.4f}")
print(f"MAE (Fehler in Bikes):   {mae:.1f}")
print(f"Weighted Score (-5/+20): {weighted_score*100:.2f}%")
print(f"Simple Score (Binary):   {simple_score:.2f}%")

# Business-Kategorisierung
lower = y_test.values * 0.95
upper = y_test.values * 1.20
ok = (y_pred_test >= lower) & (y_pred_test <= upper)
too_low = y_pred_test < lower
too_high = y_pred_test > upper

print(f"\n=== BUSINESS-KATEGORIEN ===")
print(f"✓ OK (-5% bis +20%):     {np.sum(ok)} Stunden ({np.mean(ok)*100:.1f}%)")
print(f"🔴 ZU WENIG (< -5%):     {np.sum(too_low)} Stunden ({np.mean(too_low)*100:.1f}%)")
print(f"⚠ ZU VIEL (> +20%):      {np.sum(too_high)} Stunden ({np.mean(too_high)*100:.1f}%)")


--- Finales Training auf gesamten Trainingsdaten ---

=== FINALER TEST REPORT ===
R² Score:                0.9308
MAE (Fehler in Bikes):   173.0
Weighted Score (-5/+20): 73.46%
Simple Score (Binary):   33.42%

=== BUSINESS-KATEGORIEN ===
✓ OK (-5% bis +20%):     1161 Stunden (33.4%)
🔴 ZU WENIG (< -5%):     1649 Stunden (47.5%)
⚠ ZU VIEL (> +20%):      664 Stunden (19.1%)


In [6]:
# =================================================================================
# POST-PROCESSING (Bias Correction via Multiplier)
# =================================================================================
print("\n--- Starte Post-Processing Optimierung (Gradient Boosting) ---")

# 1. Vorhersagen des besten Modells nutzen (wurden oben schon berechnet als y_pred_test)
# Falls Variable anders hieß: y_pred_raw = final_gb.predict(X_test)
y_pred_raw = y_pred_test.copy()

# 2. Suchraum definieren (Faktor 0.90 bis 1.10)
multipliers = np.arange(0.90, 1.15, 0.001)
best_m = 1.0
best_pp_score = -1.0

# 3. Optimierungsschleife
for m in multipliers:
    # Score berechnen mit bike_custom_score (deine definierte Funktion)
    score = bike_custom_score(y_test, y_pred_raw * m)
    
    if score > best_pp_score:
        best_pp_score = score
        best_m = m

# 4. Ergebnisse ausgeben
print(f"Original Weighted Score:   {bike_custom_score(y_test, y_pred_raw)*100:.2f}%")
print(f"Optimierter Score (mit PP): {best_pp_score*100:.2f}% 🚀")
print(f"Bester Multiplikator:       {best_m:.4f}")

if best_m > 1.0:
    print(f"-> Empfehlung: Hebe alle Vorhersagen um {((best_m-1)*100):.2f}% an.")
elif best_m < 1.0:
    print(f"-> Empfehlung: Senke alle Vorhersagen um {((1-best_m)*100):.2f}% ab.")

# 5. Neue Business-Kategorien anzeigen (mit PP)
y_pred_pp = y_pred_raw * best_m
lower = y_test.values * 0.95
upper = y_test.values * 1.20

ok_pp = (y_pred_pp >= lower) & (y_pred_pp <= upper)
low_pp = y_pred_pp < lower
high_pp = y_pred_pp > upper

print(f"\n=== BUSINESS-KATEGORIEN NACH OPTIMIERUNG ===")
print(f"✓ OK (-5% bis +20%): {np.sum(ok_pp)} Stunden ({np.mean(ok_pp)*100:.1f}%)")
print(f"🔴 ZU WENIG (< -5%): {np.sum(low_pp)} Stunden ({np.mean(low_pp)*100:.1f}%)")
print(f"⚠ ZU VIEL (> +20%): {np.sum(high_pp)} Stunden ({np.mean(high_pp)*100:.1f}%)")



--- Starte Post-Processing Optimierung (Gradient Boosting) ---
Original Weighted Score:   73.46%
Optimierter Score (mit PP): 83.14% 🚀
Bester Multiplikator:       1.1440
-> Empfehlung: Hebe alle Vorhersagen um 14.40% an.

=== BUSINESS-KATEGORIEN NACH OPTIMIERUNG ===
✓ OK (-5% bis +20%): 1723 Stunden (49.6%)
🔴 ZU WENIG (< -5%): 626 Stunden (18.0%)
⚠ ZU VIEL (> +20%): 1125 Stunden (32.4%)
